## Data Analytics

### Chapter 19: Feature Selection

In real-world datasets, not every feature is useful for solving a problem. Some features provide valuable information, while others are irrelevant, redundant, or simply add noise. Including unnecessary features can increase training time, reduce model accuracy, and even lead to overfitting.

#### Import Required Libraries

In [36]:
import numpy as np
import pandas as pd

#### Generate Classification Dataset

We will create two synthetic datasets with **200 samples**:

- A **House Price** dataset for regression.
- A **Spam Detection** dataset for classification.

These datasets will help us demonstrate several common feature selection techniques.

In [37]:
# House Price Dataset
n = 200
np.random.seed(42)

house_df = pd.DataFrame({
    "Area": np.random.randint(60, 250, n),
    "Bedrooms": np.random.randint(1, 6, n),
    "Age": np.random.randint(1, 35, n),
    "Distance_to_City": np.random.randint(1, 25, n),

    # Irrelevant features
    "Favorite_TV_Show": np.random.choice(
        ["Friends", "Naruto", "Football", "News", "Drama"], n
    ),
    "Children_Hobbies": np.random.choice(
        ["Music", "Gaming", "Reading", "Swimming"], n
    )
})

house_df["Price"] = (
    house_df["Area"] * 2800
    + house_df["Bedrooms"] * 18000
    - house_df["Age"] * 1200
    - house_df["Distance_to_City"] * 4000
    + np.random.normal(0, 25000, n)
).astype(int)

house_df.head(20)

,Area,Bedrooms,Age,Distance_to_City,Favorite_TV_Show,Children_Hobbies,Price
0,162,4,17,13,Friends,Swimming,461353
1,239,2,19,16,Drama,Reading,587122
2,152,3,28,13,News,Gaming,417100
3,74,1,26,14,Naruto,Music,133377
4,166,5,26,3,Friends,Gaming,498531
5,131,1,23,6,Friends,Reading,359425
6,248,1,9,18,Naruto,Music,611991
7,80,3,12,19,News,Gaming,152388
8,162,1,1,5,Friends,Swimming,411484
9,181,2,1,15,Friends,Music,496750


In [38]:
# Spam Detection Dataset
n = 200
np.random.seed(42)

spam_df = pd.DataFrame({
    "Spam_Words": np.random.randint(0, 25, n),
    "Num_Links": np.random.randint(0, 12, n),
    "Has_Attachment": np.random.randint(0, 2, n),

    # Irrelevant features
    "Favorite_Sport": np.random.choice(
        ["Football", "Basketball", "Tennis", "Chess"], n
    ),
    "Phone_Apps": np.random.randint(10, 150, n)
})

score = (
    spam_df["Spam_Words"] * 2
    + spam_df["Num_Links"] * 3
    + spam_df["Has_Attachment"] * 2
)

spam_df["Spam"] = (score > 24).astype(int)

spam_df.head(20)

,Spam_Words,Num_Links,Has_Attachment,Favorite_Sport,Phone_Apps,Spam
0,6,7,1,Chess,126,1
1,19,8,0,Football,143,1
2,14,3,0,Football,67,1
3,10,0,1,Chess,53,0
4,7,0,0,Tennis,70,0
5,20,9,1,Basketball,56,1
6,6,3,0,Basketball,89,0
7,18,11,1,Football,127,1
8,22,6,0,Tennis,29,1
9,10,1,1,Basketball,56,1


#### 19.1. Correlation-Based Selection

One of the simplest feature selection techniques is **Correlation-Based Selection**. We utilized a **correlation matrix** which measures the strength of the linear relationship between numerical variables.

- Features with **strong correlation** to the target variable are usually more useful.

- Features with **very weak or near-zero correlation** often contribute little information and may be removed.

- If two features are highly correlated with each other, keeping both may be unnecessary because they provide similar information.

In [39]:
# remove the categorical columns since correlation only applies to numerical features.
house_numeric = house_df.drop(
    columns=["Favorite_TV_Show", "Children_Hobbies"]
)

house_numeric.corr().round(2)

,Area,Bedrooms,Age,Distance_to_City,Price
Area,1.00,0.07,0.03,0.06,0.96
Bedrooms,0.07,1.00,0.06,0.03,0.21
Age,0.03,0.06,1.00,-0.04,-0.01
Distance_to_City,0.06,0.03,-0.04,1.00,-0.11
Price,0.96,0.21,-0.01,-0.11,1.00


In [40]:
# To see which features are most related to the target variable:
house_numeric.corr()["Price"].sort_values(ascending=False)

Price               1.000000
Area                0.961324
Bedrooms            0.205703
Age                -0.014001
Distance_to_City   -0.108941
Name: Price, dtype: float64

#### 19.2. Filter Methods

**Filter methods** select features by applying statistical measures **before** training a machine learning model. Instead of evaluating different models, these methods rank each feature independently according to its relationship with the target variable.

Common filter techniques include:

- Correlation coefficient (regression)
- Chi-Square test (classification)
- ANOVA F-test
- Mutual Information

One of the most widely used implementations is SelectKBest, which ranks features according to a chosen statistical score and keeps only the top k features.

In [41]:
# Selecting the Top 3 Features
from sklearn.feature_selection import SelectKBest, f_regression

X = house_numeric.drop(columns="Price")
y = house_numeric["Price"]

selector = SelectKBest(score_func=f_regression, k=3)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print(selected_features)

Index(['Area', 'Bedrooms', 'Distance_to_City'], dtype='object')


In [42]:
# To display the statistical scores
scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
})

scores.sort_values("Score", ascending=False)

,Feature,Score
0,Area,2412.189710
1,Bedrooms,8.748331
3,Distance_to_City,2.378136
2,Age,0.038819


#### 19.3. Embedded Methods

Unlike filter methods, **embedded methods** perform feature selection **during the model training process**. As the model learns, it automatically determines which features contribute the most to making accurate predictions.

Many tree-based models naturally calculate **feature importance** by measuring how much each feature helps reduce prediction error when splitting the data. Likewise, regularized linear models such as **Lasso Regression (L1 Regularization)** shrink the coefficients of less useful features toward zero, effectively removing them.

Some common embedded methods include:

- Decision Trees
- Random Forest
- Gradient Boosting
- XGBoost
- Lasso Regression

In [43]:
from sklearn.ensemble import RandomForestRegressor

X = house_numeric.drop(columns="Price")
y = house_numeric["Price"]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
0,Area,0.945802
3,Distance_to_City,0.027826
1,Bedrooms,0.015790
2,Age,0.010582


#### 16.4. Wrapper Methods

**Wrapper methods** are among the most powerful feature selection techniques. Instead of evaluating each feature individually, they repeatedly train a machine learning model using different combinations of features and select the subset that produces the best performance.

Because wrapper methods evaluate feature subsets directly, they usually achieve higher predictive performance than filter methods. However, this comes at the cost of significantly higher computational time.

The three most common wrapper methods are:

- **Forward Selection** – Starts with no features and adds one feature at a time.

- **Backward Elimination** – Starts with all features and removes the least useful feature at each step.

- **Recursive Feature Elimination (RFE)** – Recursively removes the least important feature until the desired number of features remains.

Among these methods, **Recursive Feature Elimination (RFE)** is the most widely used because it provides a good balance between accuracy and computational efficiency.

In [44]:
# Creating an Example Dataset
n = 200
np.random.seed(42)

wrapper_df = pd.DataFrame({
    "Bedrooms": np.random.randint(1, 6, n),
    "Area": np.random.randint(70, 250, n),
    "House_Color": np.random.randint(0, 5, n),
    "Family_Members": np.random.randint(1, 8, n),
    "Neighbor_Houses": np.random.randint(2, 20, n)
})

wrapper_df["Price"] = (
    wrapper_df["Area"] * 2800
    + wrapper_df["Bedrooms"] * 20000
    + wrapper_df["Family_Members"] * 3000
    + np.random.normal(0, 20000, n)
)

wrapper_df.head(20)

,Bedrooms,Area,House_Color,Family_Members,Neighbor_Houses,Price
0,4,167,3,4,8,554597.547979
1,5,208,4,6,7,726421.048663
2,3,213,1,7,3,663871.739590
3,5,166,3,3,7,529193.401847
4,5,193,1,2,19,667431.860813
5,2,139,2,2,3,418329.100755
6,3,162,0,3,19,490994.961265
7,3,72,2,7,16,282921.068373
8,3,217,3,6,3,680816.426679
9,5,233,1,3,7,777862.099515


In [45]:
# Recursive Feature Elimination (RFE)
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

X = wrapper_df.drop(columns="Price")
y = wrapper_df["Price"]

model = LinearRegression()

selector = RFE(
    estimator=model,
    n_features_to_select=3
)

selector.fit(X, y)

selected_features = X.columns[selector.support_]

print(selected_features)

Index(['Bedrooms', 'Area', 'Family_Members'], dtype='object')


In [ ]:
# To see the ranking of every feature. 
# A ranking of 1 indicates that the feature was selected by RFE.
ranking = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": selector.ranking_
})

ranking.sort_values("Ranking")

,Feature,Ranking
0,Bedrooms,1
1,Area,1
3,Family_Members,1
2,House_Color,2
4,Neighbor_Houses,3
